# Klaatch Audio Feature Extraction Pipeline

## Overview
This notebook processes Klaatch audio data from 2021-2023, extracting multiple acoustic features and storing them in a MySQL database.

## Pipeline Stages:
1. **Data Loading**: Load demographics, transcripts, and audio file metadata
2. **Data Merging**: Combine metadata from multiple sources
3. **Text Preprocessing**: Clean and standardize transcripts
4. **Feature Extraction**: Extract features using multiple methods:
   - Whisper embeddings
   - OpenSmile acoustic features
   - Librosa audio features
   - Trill embeddings
5. **Database Storage**: Store features in MySQL for downstream analysis

## Requirements
- Python 3.9+
- CUDA-capable GPU (recommended for Whisper)
- MySQL database with appropriate credentials in environment variables

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install torch torchvision torchaudio transformers librosa soundfile numpy openpyxl scikit-learn pandas pydub opensmile

In [ ]:
import os
import re
import glob
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torchaudio
import librosa
import opensmile
from pydub import AudioSegment

import mysql.connector
from mysql.connector import IntegrityError

from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2Model,
    WhisperProcessor,
    WhisperModel,
    Wav2Vec2ForSequenceClassification,
)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.neighbors import NearestNeighbors

warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# File paths
DATA_DIR = Path('.')
AUDIO_ROOT = Path('/sandata/karthik9/nwa')

# Input files
ID_MAPPING_FILE = DATA_DIR / 'Klaatch Sanitized TextCEL 2021 to 2023 (1).xlsx'
DEMOGRAPHICS_FILE = DATA_DIR / 'Demographics_Klaatch Sanitized TextCEL 2021 to 2023.xlsx'
TRANSCRIPTS_FILE = DATA_DIR / 'Klaatch_transcripts.csv'
NEW_TRANSCRIPTS_FILE = DATA_DIR / 'new_transcripts.csv'

# Feature directories
WHISPER_FEATURES_DIR = Path('/sandata/karthik9/whisper_new')
TRILL_FEATURES_DIR = Path('/sandata/karthik9/mstd_embeddings')

# Output file
PROCESSED_DATA_FILE = DATA_DIR / 'processed_klaatch_data.csv'

# Database tables
DB_TABLES = {
    'main_data': 'merged_data',
    'whisper_mean': 'feat$whisper_mean_n$merged_data$message_id',
    'whisper_median': 'feat$whisper_median_n$merged_data$message_id',
    'opensmile': 'feat$opensmile_n$merged_data$message_id',
    'librosa': 'feat$librosa_n$merged_data$message_id',
    'trill': 'feat$trill_mstd$merged_data$message_id',
    'audio_processing': 'new_audio_processing_status'
}

print("Configuration loaded successfully")
print(f"Audio root: {AUDIO_ROOT}")
print(f"Whisper features: {WHISPER_FEATURES_DIR}")

## 3. Database Connection

In [ ]:
def connect_to_db():
    """
    Establish connection to MySQL database using environment variables.
    
    Required environment variables:
    - DB_HOST
    - DB_DATABASE
    - DB_USERNAME
    - DB_PASSWORD
    """
    return mysql.connector.connect(
        host=os.getenv('DB_HOST'),
        database=os.getenv('DB_DATABASE'),
        user=os.getenv('DB_USERNAME'),
        password=os.getenv('DB_PASSWORD')
    )

# Test connection
try:
    conn = connect_to_db()
    print("Database connection successful")
    conn.close()
except Exception as e:
    print(f"Database connection failed: {e}")

## 4. Utility Functions

In [ ]:
def convert_to_wav(audio_file_path):
    """
    Convert audio file to WAV format if not already in WAV.
    
    Args:
        audio_file_path (str): Path to audio file
        
    Returns:
        str: Path to WAV file
    """
    if not audio_file_path.endswith('.wav'):
        print(f"Converting {audio_file_path} to WAV format...")
        audio = AudioSegment.from_file(audio_file_path)
        wav_path = audio_file_path.rsplit('.', 1)[0] + '.wav'
        audio.export(wav_path, format='wav')
        return wav_path
    return audio_file_path


def compute_mean_median(file_path):
    """
    Compute per-dimension mean and median for a NumPy array.
    
    Args:
        file_path (str): Path to .npy file
        
    Returns:
        tuple: (mean_array, median_array)
    """
    data = np.load(file_path)
    
    if data.ndim == 1:
        return data, data
    
    mean = np.mean(data, axis=0)
    median = np.median(data, axis=0)
    
    return mean, median


def is_file_processed(filename, table_name, connection):
    """
    Check if a file has already been processed and stored in database.
    
    Args:
        filename (str): Filename to check
        table_name (str): Database table name
        connection: MySQL connection object
        
    Returns:
        bool: True if already processed
    """
    query = f"SELECT COUNT(*) FROM {table_name} WHERE filename = %s"
    cursor = connection.cursor()
    cursor.execute(query, (filename,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0


def insert_data(df, table_name):
    """
    Insert main dataset into the specified database table.
    
    Inserts records with: message_id, message, KlaatchID, Date, CEL_Total, 
    CELVAL1, CELVAL2, CELVAL3, and Age columns.
    
    Args:
        df (pd.DataFrame): DataFrame containing the data to insert
        table_name (str): Name of the database table (e.g., 'merged_data')
    """
    connection = connect_to_db()
    cursor = connection.cursor()

    insert_query = f"""
        INSERT IGNORE INTO {table_name} (
            message_id, message, KlaatchID, Date, CEL_Total, CELVAL1, CELVAL2, CELVAL3, 
            Age
        ) VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s)
    """

    # Convert DataFrame to list of tuples with required columns
    required_cols = ['Filename', 'Text', 'KlaatchID', 'Date', 'CEL Total', 
                     'CELVAL1', 'CELVAL2', 'CELVAL3', 'Age']
    
    # Check if all columns exist
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        print(f"Warning: Missing columns {missing_cols}, cannot insert data")
        cursor.close()
        connection.close()
        return
    
    data = df[required_cols].values.tolist()
    
    try:
        cursor.executemany(insert_query, data)
        connection.commit()
        print(f"Inserted {cursor.rowcount} rows successfully into table {table_name}.")
    except mysql.connector.Error as err:
        print(f"Error: {err}")
    finally:
        cursor.close()
        connection.close()


def insert_feature_table_into_db(connection, dataframe, table_name):
    """
    Insert feature dataframe into MySQL database.
    
    Args:
        connection: MySQL connection object
        dataframe (pd.DataFrame): Data to insert
        table_name (str): Target table name
    """
    cursor = connection.cursor()
    
    columns = ', '.join(dataframe.columns)
    placeholders = ', '.join(['%s'] * len(dataframe.columns))
    
    insert_query = f"""
        INSERT INTO {table_name} ({columns}) 
        VALUES ({placeholders})
    """
    
    inserted_count = 0
    
    for _, row in dataframe.iterrows():
        try:
            cursor.execute(insert_query, tuple(row))
            inserted_count += 1
        except IntegrityError:
            continue
    
    connection.commit()
    cursor.close()
    
    print(f"Inserted {inserted_count}/{len(dataframe)} rows into {table_name}")


print("Utility functions loaded")

## 4b. Audio Preprocessing Functions (Enhanced)

In [ ]:
def preprocess_audio(audio_file_path, target_sample_rate=16000):
    """
    Preprocess audio file: convert to WAV and resample to target sample rate.
    
    Args:
        audio_file_path (str): Path to audio file
        target_sample_rate (int): Target sample rate (default: 16000 Hz)
        
    Returns:
        tuple: (waveform, sample_rate)
    """
    # Convert to WAV if necessary
    if not audio_file_path.endswith('.wav'):
        print(f"Converting {audio_file_path} to WAV format...")
        audio = AudioSegment.from_file(audio_file_path)
        new_file_path = os.path.splitext(audio_file_path)[0] + ".wav"
        audio.export(new_file_path, format="wav")
        audio_file_path = new_file_path
        print(f"Conversion complete: {audio_file_path}")
    
    # Load audio with torchaudio
    waveform, sampling_rate = torchaudio.load(audio_file_path)
    
    # Resample if necessary
    if sampling_rate != target_sample_rate:
        print(f"Resampling from {sampling_rate} Hz to {target_sample_rate} Hz...")
        resample_transform = torchaudio.transforms.Resample(
            orig_freq=sampling_rate, 
            new_freq=target_sample_rate
        )
        waveform = resample_transform(waveform)
    
    return waveform, target_sample_rate

print("Enhanced audio preprocessing function loaded")

## 5. Data Loading and Merging

In [ ]:
# Load ID mapping
print("Loading ID mapping...")
oldid_newid_df = pd.read_excel(ID_MAPPING_FILE, sheet_name='Klaatch Sanitized')
print(f"Loaded {len(oldid_newid_df)} ID mappings")

oldid_newid_df.head()

In [ ]:
# Load demographics
print("Loading demographics...")
demographics_df = pd.read_excel(DEMOGRAPHICS_FILE)
print(f"Loaded demographics for {len(demographics_df)} participants")
print(f"Columns: {list(demographics_df.columns)}")

demographics_df.head()

In [ ]:
# Load transcripts
print("Loading transcripts...")
transcripts_df = pd.read_csv(TRANSCRIPTS_FILE)
print(f"Loaded {len(transcripts_df)} transcripts")

# Clean and standardize IDs
transcripts_df['New ID'] = pd.to_numeric(transcripts_df['New ID'], errors='coerce').replace(0, np.nan)
transcripts_df['New ID'] = transcripts_df['New ID'].astype(str)
transcripts_df['Old ID'] = transcripts_df['Old ID'].astype(int).astype(str)

print(f"Transcripts with New ID: {transcripts_df['New ID'].notna().sum()}")
transcripts_df.head()

In [ ]:
# Merge transcripts with demographics
print("Merging transcripts with demographics...")
metadata_df = pd.merge(
    transcripts_df, 
    demographics_df, 
    left_on='New ID', 
    right_on='New ID', 
    how='left'
)

print(f"Merged dataset: {len(metadata_df)} records")
print(f"Missing demographics: {metadata_df['Age'].isna().sum()} records")

metadata_df.info()

## 6. Audio File Discovery

In [ ]:
print(f"Scanning audio files in {AUDIO_ROOT}...")

file_data = []

for root, dirs, files in os.walk(AUDIO_ROOT):
    for file in files:
        if file.endswith('.mp3'):
            file_path = os.path.join(root, file)
            
            # Extract metadata from filename
            # Expected format: {ID}_{Date}_{Time}.mp3
            parts = file.replace('.mp3', '').split('_')
            
            if len(parts) >= 2:
                klaatch_id = parts[0]
                date = parts[1] if len(parts) > 1 else 'unknown'
                
                file_data.append({
                    'Filename': file.replace('.mp3', ''),
                    'Filepath': file_path,
                    'Original_Filename': file,
                    'KlaatchID': klaatch_id,
                    'Date': date
                })

df_audio = pd.DataFrame(file_data)
df_audio = df_audio.drop_duplicates(subset='Filename', keep='first')

print(f"Found {len(df_audio)} unique audio files")
print(f"Unique participants: {df_audio['KlaatchID'].nunique()}")

df_audio.head(10)

In [ ]:
# Extract date from metadata filenames for consistency
metadata_df['Date'] = metadata_df['Filename'].apply(lambda x: x.split('_')[1] if '_' in str(x) else 'unknown')

# Merge metadata with audio files
print("Merging metadata with audio file paths...")
merged_df = pd.merge(metadata_df, df_audio, on='Filename', how='left')

print(f"Merged dataset: {len(merged_df)} records")
print(f"Records with audio files: {merged_df['Filepath'].notna().sum()}")
print(f"Records missing audio: {merged_df['Filepath'].isna().sum()}")

merged_df.head()

In [ ]:
# Load additional transcripts if available
if NEW_TRANSCRIPTS_FILE.exists():
    print("Loading additional transcripts...")
    new_transcripts = pd.read_csv(NEW_TRANSCRIPTS_FILE)
    print(f"Loaded {len(new_transcripts)} additional transcripts")
    
    # Merge with main dataset
    # Update records that were missing New ID
    merged_df = merged_df.merge(
        new_transcripts,
        on='Filename',
        how='left',
        suffixes=('', '_new')
    )
    print(f"Updated dataset: {len(merged_df)} records")
else:
    print(f"No additional transcripts file found at {NEW_TRANSCRIPTS_FILE}")

## 7. Text Preprocessing

In [ ]:
print("Preprocessing text transcripts with comprehensive pattern removal...")

# Create a copy for text processing
stopwords_merged_df = merged_df.copy()

# Define comprehensive patterns to remove
patterns_to_remove = [
    'lisa\w*',
    'lisa',
    'speaker1', 'speaker2',
    'strongly agree',
    'strongly disagree',
    'strongly_agree',
    'strongly_disagree',
    'agree\w*',          # Matches agree, agreed, agreement, agreeable, agreeing
    'disagree\w*',       # Matches disagree, disagreed, disagreement, disagreeing
    'neutral',
    'Speaker',
    'strongly agree\w*', # Matches strongly agree variations
    'strongly disagree\w*',
    'strongly_agree\w*',
    'strongly_disagree\w*',
    'Speaker\w*',
    'nn\w*',
    'nnspeaker\w*',
    '!', '\$', ',', '\.', '\.\.', '\.\s*mm hmm', '\.\.',
    '0', '00 \]', '0:00', '0:00:00',
    '1', '10', '100', '11', '12', '13', '14', '15', '16', '18',
    '2', '20', '20 years', '25',
    '3', '30', '4', '40',
    '5', '5 to 10', '50', '90',
    ':', ': 00 \]', '\?', '\]', '\] hello', '\] hello \.',
    "don't\w*", 'not\w*', 'strongly\w*', 'definitely\w*', 'neither\w*'
]

# Regex pattern for timestamps like [00:24:00]
timestamp_pattern = r'\[\d{2}:\d{2}:\d{2}\]'

# Separate regex patterns from literal strings
escaped_patterns = []
for pattern in patterns_to_remove:
    if '\\w' in pattern:
        escaped_patterns.append(pattern)  # Keep regex operators
    else:
        escaped_patterns.append(re.escape(pattern))  # Escape others

# Build combined pattern
words_regex = r'\b(?:' + '|'.join(escaped_patterns) + r')\b'
full_regex_pattern = f'(?:{words_regex})|(?:{timestamp_pattern})'

# Apply text cleaning
if 'Text' in stopwords_merged_df.columns:
    stopwords_merged_df['Text'] = stopwords_merged_df['Text'].str.replace(
        full_regex_pattern, 
        '', 
        regex=True, 
        flags=re.IGNORECASE
    )
    
    # Clean up extra whitespace
    stopwords_merged_df['Text'] = stopwords_merged_df['Text'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # Calculate text length
    stopwords_merged_df['Text_Length'] = stopwords_merged_df['Text'].str.len()
    
    print(f"Original text samples: {merged_df['Text'].notna().sum()}")
    print(f"Cleaned text samples: {(stopwords_merged_df['Text_Length'] > 0).sum()}")
    print(f"Empty after cleaning: {(stopwords_merged_df['Text_Length'] == 0).sum()}")
    
    # Update merged_df with cleaned text
    merged_df = stopwords_merged_df.copy()
    
else:
    print("Warning: 'Text' column not found in merged dataset")
    merged_df['Text_Length'] = 0

print(f"\nText cleaning complete with {len(patterns_to_remove)} patterns removed")
merged_df[['Filename', 'Text_Length']].describe()

In [ ]:
# Save processed data
print(f"Saving processed data to {PROCESSED_DATA_FILE}...")
merged_df.to_csv(PROCESSED_DATA_FILE, index=False)
print(f"Saved {len(merged_df)} records")

# Create a working copy for feature extraction
feature_df = merged_df.copy()
print(f"\nDataset ready for feature extraction: {len(feature_df)} records")

In [ ]:
# Optional: Insert main dataset into database
# Uncomment the line below to insert the merged dataset into the 'merged_data' table
# insert_data(merged_df, DB_TABLES['main_data'])

print("To insert main dataset into database, uncomment the insert_data() call above")

## 8. Statistical Analysis

In [ ]:
# Group statistics by participant
if 'CEL Total' in merged_df.columns and 'KlaatchID' in merged_df.columns:
    print("Computing participant-level statistics...")
    
    grouped_stats = merged_df.groupby('KlaatchID')['CEL Total'].agg(['mean', 'std', 'count', 'min', 'max'])
    grouped_stats.columns = ['CEL_Mean', 'CEL_Std', 'Sample_Count', 'CEL_Min', 'CEL_Max']
    
    print(f"\nParticipants: {len(grouped_stats)}")
    print(f"Avg samples per participant: {grouped_stats['Sample_Count'].mean():.1f}")
    
    grouped_stats.describe()

## 9. Whisper Feature Extraction

In [ ]:
print("Setting up Whisper feature extraction...")

WHISPER_FEATURES_DIR.mkdir(parents=True, exist_ok=True)

def is_whisper_processed(filename):
    """
    Check if Whisper features exist for a given file.
    
    Args:
        filename (str): Base filename without extension
        
    Returns:
        bool: True if features exist
    """
    whisper_file = WHISPER_FEATURES_DIR / f"{filename}_whisper.npy"
    return whisper_file.exists()

# Check existing Whisper features
whisper_files = list(WHISPER_FEATURES_DIR.glob('*_whisper.npy'))
print(f"Found {len(whisper_files)} existing Whisper feature files")

if len(whisper_files) > 0:
    # Extract filenames from Whisper features
    whisper_filenames = [f.stem.replace('_whisper', '') for f in whisper_files]
    
    # Mark processed files in dataframe
    feature_df['Whisper_Processed'] = feature_df['Filename'].isin(whisper_filenames)
    feature_df['Whisper_Path'] = feature_df['Filename'].apply(
        lambda x: str(WHISPER_FEATURES_DIR / f"{x}_whisper.npy") if is_whisper_processed(x) else None
    )
    
    print(f"Whisper features available for {feature_df['Whisper_Processed'].sum()} files")
else:
    print("No Whisper features found. Run feature extraction first.")

## 9b. Whisper Real-Time Feature Extraction (NEW)

In [ ]:
def is_whisper_file_processed(filename):
    """
    Check if Whisper features already exist for a file.
    
    Args:
        filename (str): Original filename (e.g., 'audio.mp3')
        
    Returns:
        tuple: (is_processed: bool, features_file_path: str)
    """
    base_name = filename.rsplit('.mp3', 1)[0]
    whisper_features_file = os.path.join(WHISPER_FEATURES_DIR, f"{base_name}_whisper_features.npy")
    return os.path.exists(whisper_features_file), whisper_features_file


def save_whisper_features_to_file(features, filename):
    """
    Save Whisper extracted features to a file.
    
    Args:
        features (np.ndarray): Extracted features
        filename (str): Original filename
        
    Returns:
        str: Path to saved features file
    """
    base_name = filename.rsplit('.mp3', 1)[0]
    
    # Ensure the directory exists
    if not os.path.exists(WHISPER_FEATURES_DIR):
        os.makedirs(WHISPER_FEATURES_DIR)
    
    # Create the full file path
    features_file_path = os.path.join(WHISPER_FEATURES_DIR, f"{base_name}_whisper_features.npy")
    
    # Save the features as a NumPy file
    np.save(features_file_path, features)
    
    return features_file_path


def extract_whisper_features(waveform, sample_rate):
    """
    Extract Whisper features from audio waveform.
    
    Args:
        waveform (torch.Tensor): Audio waveform
        sample_rate (int): Sample rate
        
    Returns:
        np.ndarray: Extracted features of shape (sequence_length, hidden_size)
    """
    print("Loading Whisper processor and model...")
    processor = WhisperProcessor.from_pretrained('openai/whisper-base')
    model = WhisperModel.from_pretrained('openai/whisper-base')
    model.eval()

    print("Processing audio and preparing input tensors for Whisper...")
    input_features = processor(
        waveform.squeeze().numpy(), 
        sampling_rate=sample_rate, 
        return_tensors="pt"
    ).input_features

    print("Extracting Whisper features...")
    with torch.no_grad():
        encoder_outputs = model.get_encoder()(input_features)
        features = encoder_outputs.last_hidden_state

    print(f"Whisper features extracted. Shape: {features.shape}")

    # Convert features to NumPy array
    features_np = features.squeeze().numpy()
    return features_np


def process_and_store_whisper_features(merged_df):
    """
    Process audio files and extract Whisper features.
    
    Args:
        merged_df (pd.DataFrame): DataFrame with audio file information
        
    Returns:
        pd.DataFrame: DataFrame with processed files and feature paths
    """
    processed_files = []

    for index, row in merged_df.iterrows():
        audio_file_path = row['Filepath']
        filename = row['Filename']
        original_filename = row['Original_Filename']
        
        # Check if the file is already processed
        is_processed, features_file_path = is_whisper_file_processed(original_filename)
        
        if is_processed:
            print(f"Skipping {filename}, already processed.")
            processed_files.append({
                "Filename": filename, 
                "features_file_path": features_file_path
            })
            continue
        
        try:
            print(f"Processing: {filename} ({original_filename})")
            
            # Preprocess and extract Whisper features
            waveform, sample_rate = preprocess_audio(audio_file_path)
            whisper_features = extract_whisper_features(waveform, sample_rate)
            
            print(f"Feature vector size for {original_filename}: {whisper_features.shape}")
            
            # Save the extracted Whisper features to a file
            whisper_features_file_path = save_whisper_features_to_file(
                whisper_features, 
                original_filename
            )
            
            processed_files.append({
                "Filename": filename,
                "features_file_path": whisper_features_file_path
            })
            
            print(f"Whisper features for {filename} stored successfully in {whisper_features_file_path}.")

        except Exception as e:
            print(f"Error processing {audio_file_path} with Whisper: {e}")

    processed_df = pd.DataFrame(processed_files, columns=["Filename", "features_file_path"])
    return processed_df

print("Whisper extraction functions loaded")

In [ ]:
# Uncomment to extract Whisper features from audio files
# This will process all audio files and extract Whisper features
# WARNING: This can take a long time for large datasets

# whisper_processed_df = process_and_store_whisper_features(merged_df)
# print(f"Processed {len(whisper_processed_df)} files")

print("To extract Whisper features from audio, uncomment the code above")

In [ ]:
# Load and process Whisper features
if 'Whisper_Path' in feature_df.columns:
    print("Computing Whisper feature statistics...")
    
    whisper_data = []
    
    for idx, row in feature_df[feature_df['Whisper_Processed'] == True].iterrows():
        filepath = row['Whisper_Path']
        filename = row['Filename']
        
        if filepath and os.path.exists(filepath):
            mean_feat, median_feat = compute_mean_median(filepath)
            
            whisper_data.append({
                'group_id': filename,
                'mean_features': mean_feat,
                'median_features': median_feat
            })
    
    if whisper_data:
        print(f"Processed Whisper features for {len(whisper_data)} files")
        
        # Create mean features dataframe
        mean_records = []
        for record in whisper_data:
            for i, val in enumerate(record['mean_features']):
                mean_records.append({
                    'group_id': record['group_id'],
                    'feat': f'whisper_mean_{i}',
                    'value': float(val)
                })
        
        whisper_mean_df = pd.DataFrame(mean_records)
        print(f"Whisper mean features: {len(whisper_mean_df)} records")
        
        # Create median features dataframe
        median_records = []
        for record in whisper_data:
            for i, val in enumerate(record['median_features']):
                median_records.append({
                    'group_id': record['group_id'],
                    'feat': f'whisper_median_{i}',
                    'value': float(val)
                })
        
        whisper_median_df = pd.DataFrame(median_records)
        print(f"Whisper median features: {len(whisper_median_df)} records")

In [ ]:
# Insert Whisper features into database
if 'whisper_mean_df' in locals() and len(whisper_mean_df) > 0:
    print("Inserting Whisper features into database...")
    
    try:
        connection = connect_to_db()
        
        print("Inserting mean features...")
        insert_feature_table_into_db(connection, whisper_mean_df, DB_TABLES['whisper_mean'])
        
        print("Inserting median features...")
        insert_feature_table_into_db(connection, whisper_median_df, DB_TABLES['whisper_median'])
        
        connection.close()
        print("Whisper features inserted successfully")
        
    except Exception as e:
        print(f"Error inserting Whisper features: {e}")
else:
    print("No Whisper features to insert")

## 10. OpenSmile Feature Extraction

## 10b. OpenSmile Real-Time Feature Extraction (NEW)

In [ ]:
def is_file_processed_opensmile_from_db(filename, connection):
    """
    Check if file has been processed with OpenSmile (checks new_audio_features table).
    
    Args:
        filename (str): Filename to check
        connection: MySQL connection
        
    Returns:
        bool: True if processed
    """
    query = "SELECT COUNT(*) FROM new_audio_features WHERE filename = %s"
    cursor = connection.cursor()
    cursor.execute(query, (filename,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0


def insert_features_into_db_opensmile(filename, cel_total, features, connection):
    """
    Insert OpenSmile feature data into MySQL with features stored as JSON.
    
    Args:
        filename (str): Audio filename
        cel_total (float): CEL Total value
        features (dict): Dictionary of feature values
        connection: MySQL connection
    """
    # Convert features dictionary to JSON format
    features_json = json.dumps(features)
    
    # Build the SQL INSERT statement
    query = """
        INSERT INTO new_audio_features (filename, cel_total, feature_data)
        VALUES (%s, %s, %s)
    """
    
    # Execute the INSERT statement
    cursor = connection.cursor()
    cursor.execute(query, (filename, cel_total, features_json))
    connection.commit()
    cursor.close()

print("OpenSmile extraction functions loaded")

In [ ]:
# Uncomment to extract OpenSmile features from audio files
# This processes each audio file and extracts eGeMAPSv02 features (88 features)
# WARNING: This can take a long time for large datasets

# connection = connect_to_db()
# processed_count = 0
# skipped_count = 0

# for idx, row in merged_df.iterrows():
#     filename = row['Filename']
#     
#     # Check if this file has already been processed
#     if is_file_processed_opensmile_from_db(filename, connection):
#         print(f"Skipping {filename}, already processed.")
#         skipped_count += 1
#         continue
#     
#     # Get file path and preprocess the audio file
#     audio_path = row['Filepath']
#     audio_path = convert_to_wav(audio_path)
#     waveform, sample_rate = preprocess_audio(audio_path)
#     
#     # Process the preprocessed audio file with OpenSmile
#     features_df = smile.process_file(audio_path)
#     
#     # Extract features as a dictionary
#     features = features_df.iloc[0].to_dict()
#     
#     # Insert the features into the database as JSON
#     insert_features_into_db_opensmile(filename, row['CEL Total'], features, connection)
#     print(f"Processed and stored features for {filename}")
#     processed_count += 1

# connection.close()
# print(f"OpenSmile extraction complete: {processed_count} processed, {skipped_count} skipped")

print("To extract OpenSmile features from audio, uncomment the code above")

In [ ]:
print("Initializing OpenSmile...")

smile = opensmile.Smile(
    feature_set=opensmile.FeatureSet.eGeMAPSv02,
    feature_level=opensmile.FeatureLevel.Functionals,
)

print(f"OpenSmile initialized with {smile.feature_names.shape[0]} features")
print(f"Feature set: eGeMAPSv02 (Geneva Minimalistic Acoustic Parameter Set)")

In [ ]:
# Check which files have been processed with OpenSmile
def is_file_processed_opensmile(filename, connection):
    """
    Check if file has been processed with OpenSmile.
    
    Args:
        filename (str): Filename to check
        connection: MySQL connection
        
    Returns:
        bool: True if processed
    """
    query = "SELECT COUNT(*) FROM new_audio_processing_status WHERE filename = %s AND opensmile_processed = 1"
    cursor = connection.cursor()
    cursor.execute(query, (filename,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0

print("Checking OpenSmile processing status...")
# This would query the database to get status

## 11b. Librosa Real-Time Feature Extraction (NEW)

In [ ]:
def is_file_processed_librosa_from_db(filename, connection):
    """
    Check if file has been processed with Librosa (checks new_librosa_features table).
    
    Args:
        filename (str): Filename to check
        connection: MySQL connection
        
    Returns:
        bool: True if processed
    """
    query = "SELECT COUNT(*) FROM new_librosa_features WHERE filename = %s"
    cursor = connection.cursor()
    cursor.execute(query, (filename,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0


def convert_features_to_float(features):
    """
    Convert all feature values to Python float.
    
    Args:
        features (dict): Dictionary with feature values
        
    Returns:
        dict: Dictionary with float values
    """
    return {key: float(value) for key, value in features.items()}


def insert_features_into_db_librosa(filename, cel_total, features, connection):
    """
    Insert Librosa features into MySQL with features stored in individual columns.
    
    Extracts 38 features:
    - 13 MFCCs (Mel-frequency cepstral coefficients)
    - 12 Chroma features
    - 7 Spectral contrast bands
    - 6 Tonnetz features
    
    Args:
        filename (str): Audio filename
        cel_total (float): CEL Total value
        features (dict): Dictionary containing individual feature values
        connection: MySQL connection
    """
    query = """
        INSERT INTO new_librosa_features (
            filename, cel_total,
            mfcc_1, mfcc_2, mfcc_3, mfcc_4, mfcc_5, mfcc_6, mfcc_7, mfcc_8, mfcc_9, mfcc_10, mfcc_11, mfcc_12, mfcc_13,
            chroma_1, chroma_2, chroma_3, chroma_4, chroma_5, chroma_6, chroma_7, chroma_8, chroma_9, chroma_10, chroma_11, chroma_12,
            spectral_contrast_1, spectral_contrast_2, spectral_contrast_3, spectral_contrast_4, spectral_contrast_5, spectral_contrast_6, spectral_contrast_7,
            tonnetz_1, tonnetz_2, tonnetz_3, tonnetz_4, tonnetz_5, tonnetz_6
        )
        VALUES (
            %s, %s,
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s, %s,
            %s, %s, %s, %s, %s, %s
        )
    """
    
    params = (
        filename,
        cel_total,
        features.get('mfcc_1'), features.get('mfcc_2'), features.get('mfcc_3'),
        features.get('mfcc_4'), features.get('mfcc_5'), features.get('mfcc_6'),
        features.get('mfcc_7'), features.get('mfcc_8'), features.get('mfcc_9'),
        features.get('mfcc_10'), features.get('mfcc_11'), features.get('mfcc_12'),
        features.get('mfcc_13'),
        features.get('chroma_1'), features.get('chroma_2'), features.get('chroma_3'),
        features.get('chroma_4'), features.get('chroma_5'), features.get('chroma_6'),
        features.get('chroma_7'), features.get('chroma_8'), features.get('chroma_9'),
        features.get('chroma_10'), features.get('chroma_11'), features.get('chroma_12'),
        features.get('spectral_contrast_1'), features.get('spectral_contrast_2'),
        features.get('spectral_contrast_3'), features.get('spectral_contrast_4'),
        features.get('spectral_contrast_5'), features.get('spectral_contrast_6'),
        features.get('spectral_contrast_7'),
        features.get('tonnetz_1'), features.get('tonnetz_2'), features.get('tonnetz_3'),
        features.get('tonnetz_4'), features.get('tonnetz_5'), features.get('tonnetz_6')
    )
    
    try:
        cursor = connection.cursor()
        cursor.execute(query, params)
        connection.commit()
        cursor.close()
    except IntegrityError as e:
        print(f"IntegrityError while inserting {filename}: {e}")
        connection.rollback()
    except Exception as e:
        print(f"Unexpected error while inserting {filename}: {e}")
        connection.rollback()


def process_audio_files_librosa(merged_df, connection):
    """
    Process audio files and extract Librosa features.
    
    Extracts:
    - 13 MFCCs (Mel-frequency cepstral coefficients)
    - 12 Chroma features (pitch class profiles)
    - 7 Spectral contrast bands
    - 6 Tonnetz features (tonal centroid features)
    
    Args:
        merged_df (pd.DataFrame): DataFrame with audio file information
        connection: MySQL connection
    """
    processed_count = 0
    skipped_count = 0
    error_count = 0
    
    for idx, row in merged_df.iterrows():
        filename = row['Filename']
        
        # Check if this file has already been processed
        if is_file_processed_librosa_from_db(filename, connection):
            print(f"Skipping {filename}, already processed.")
            skipped_count += 1
            continue
        
        # Get file path and load the audio file
        audio_path = row['Filepath']
        try:
            y, sr = librosa.load(audio_path, sr=None)
        except Exception as e:
            print(f"Error loading {audio_path}: {e}")
            error_count += 1
            continue
    
        # Extract features using Librosa
        try:
            mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            chroma = librosa.feature.chroma_stft(y=y, sr=sr)
            spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
            tonnetz = librosa.feature.tonnetz(y=librosa.effects.harmonic(y), sr=sr)
        except Exception as e:
            print(f"Error extracting features for {filename}: {e}")
            error_count += 1
            continue
    
        # Aggregate features by taking the mean across time frames
        mfccs_mean = mfccs.mean(axis=1)
        chroma_mean = chroma.mean(axis=1)
        spectral_contrast_mean = spectral_contrast.mean(axis=1)
        tonnetz_mean = tonnetz.mean(axis=1)
    
        # Create a features dictionary with individual feature values
        features = {
            "mfcc_1": mfccs_mean[0], "mfcc_2": mfccs_mean[1], "mfcc_3": mfccs_mean[2],
            "mfcc_4": mfccs_mean[3], "mfcc_5": mfccs_mean[4], "mfcc_6": mfccs_mean[5],
            "mfcc_7": mfccs_mean[6], "mfcc_8": mfccs_mean[7], "mfcc_9": mfccs_mean[8],
            "mfcc_10": mfccs_mean[9], "mfcc_11": mfccs_mean[10], "mfcc_12": mfccs_mean[11],
            "mfcc_13": mfccs_mean[12],
            "chroma_1": chroma_mean[0], "chroma_2": chroma_mean[1], "chroma_3": chroma_mean[2],
            "chroma_4": chroma_mean[3], "chroma_5": chroma_mean[4], "chroma_6": chroma_mean[5],
            "chroma_7": chroma_mean[6], "chroma_8": chroma_mean[7], "chroma_9": chroma_mean[8],
            "chroma_10": chroma_mean[9], "chroma_11": chroma_mean[10], "chroma_12": chroma_mean[11],
            "spectral_contrast_1": spectral_contrast_mean[0],
            "spectral_contrast_2": spectral_contrast_mean[1],
            "spectral_contrast_3": spectral_contrast_mean[2],
            "spectral_contrast_4": spectral_contrast_mean[3],
            "spectral_contrast_5": spectral_contrast_mean[4],
            "spectral_contrast_6": spectral_contrast_mean[5],
            "spectral_contrast_7": spectral_contrast_mean[6],
            "tonnetz_1": tonnetz_mean[0], "tonnetz_2": tonnetz_mean[1],
            "tonnetz_3": tonnetz_mean[2], "tonnetz_4": tonnetz_mean[3],
            "tonnetz_5": tonnetz_mean[4], "tonnetz_6": tonnetz_mean[5]
        }
        
        # Convert feature values to Python floats
        features = convert_features_to_float(features)
    
        # Insert the features into the database
        insert_features_into_db_librosa(filename, row['CEL Total'], features, connection)
        print(f"Processed and stored features for {filename}")
        processed_count += 1
    
    print(f"\nLibrosa extraction complete:")
    print(f"  Processed: {processed_count}")
    print(f"  Skipped: {skipped_count}")
    print(f"  Errors: {error_count}")

print("Librosa extraction functions loaded")

In [ ]:
# Uncomment to extract Librosa features from audio files
# This extracts 38 acoustic features (MFCCs, Chroma, Spectral Contrast, Tonnetz)
# WARNING: This can take a long time for large datasets

# try:
#     connection = connect_to_db()
#     process_audio_files_librosa(merged_df, connection)
# finally:
#     connection.close()

print("To extract Librosa features from audio, uncomment the code above")

In [ ]:
# Retrieve existing OpenSmile features from database
def retrieve_features_from_opensmile_db():
    """
    Retrieve OpenSmile features from database.
    
    Returns:
        pd.DataFrame: Features dataframe
    """
    connection = connect_to_db()
    
    query = "SELECT * FROM new_audio_processing_status WHERE opensmile_processed = 1"
    df = pd.read_sql(query, connection)
    
    connection.close()
    return df

try:
    opensmile_df = retrieve_features_from_opensmile_db()
    print(f"Retrieved {len(opensmile_df)} OpenSmile feature records from database")
    
    if len(opensmile_df) > 0:
        print(f"Columns: {list(opensmile_df.columns[:10])}...")  # Show first 10 columns
        
except Exception as e:
    print(f"Could not retrieve OpenSmile features: {e}")
    opensmile_df = pd.DataFrame()

## 12b. Alternative Trill Processing (With STD Embeddings)

In [ ]:
# Alternative Trill file discovery using *_std_trill_embedding.npy pattern
# Uncomment this section if your Trill files use the "_std_" naming convention

# directory_path = '/sandata/karthik9/mstd_embeddings/'
# pattern = os.path.join(directory_path, "*_std_trill_embedding.npy")
# file_list = glob.glob(pattern)

# # Create a dictionary mapping from filename to full path
# filename_to_path = {}
# for file_path in file_list:
#     base_name = os.path.basename(file_path)
#     key = base_name.split('_std_trill_embedding.npy')[0]
#     filename_to_path[key] = file_path

# print(f"Found {len(filename_to_path)} Trill STD embedding files")

# # Remove .mp3 extension from Original_Filename if present
# stopwords_merged_df['Original_Filename_Clean'] = stopwords_merged_df['Original_Filename'].str.replace(r'\.mp3$', '', regex=True)

# # Map to DataFrame
# stopwords_merged_df['Trill_STD_Path'] = stopwords_merged_df['Original_Filename_Clean'].map(filename_to_path)

# # Check which rows didn't find a match
# missing = stopwords_merged_df[stopwords_merged_df['Trill_STD_Path'].isna()]
# if not missing.empty:
#     print(f"No matching file found for {len(missing)} filenames")
# else:
#     print("All filenames matched a file path")

print("Alternative Trill STD processing code available (commented out)")

In [ ]:
def load_numpy_to_columns(npy_path):
    """
    Load 1024-dimensional Trill embedding and expand into dictionary of columns.
    
    Args:
        npy_path (str): Path to .npy file
        
    Returns:
        dict: Dictionary with keys 'dim0' through 'dim1023'
    """
    array = np.load(npy_path)
    return {f"dim{i}": array[i] for i in range(1024)}


# Example usage for expanding Trill embeddings (commented out)
# This expands 1024-dimensional embeddings into separate columns

# trill_mean = stopwords_merged_df[['Filename', 'Trill_STD_Path', 'KlaatchID']].copy()

# # Apply the function to each row and expand columns
# expanded_columns = trill_mean['Trill_STD_Path'].apply(load_numpy_to_columns).apply(pd.Series)

# # Concatenate original DataFrame with expanded columns
# trill_mean = pd.concat([trill_mean, expanded_columns], axis=1)

# # Drop the path column if no longer needed
# trill_mean.drop(columns=['Trill_STD_Path', 'KlaatchID'], inplace=True)
# trill_mean = trill_mean.rename(columns={'Filename': 'group_id'})

# print(f"Trill embeddings expanded into {len(expanded_columns.columns)} columns")

print("Trill expansion function loaded (load_numpy_to_columns)")

## 14. Wav2Vec2 Feature Extraction (Optional)

In [ ]:
# Wav2Vec2 Feature Extraction (Optional)
# Uncomment this section to extract Wav2Vec2 features from Facebook's model
# Uses chunking to handle long audio files that might exceed GPU memory

# FEATURES_DIR = '/sandata/karthik9/wav2vec2_features'

# def extract_wav2vec2_features(waveform, sample_rate, chunk_duration=5.0):
#     """
#     Extract Wav2Vec2 features with chunking for long audio.
#     
#     Args:
#         waveform (torch.Tensor): Audio waveform
#         sample_rate (int): Sample rate
#         chunk_duration (float): Duration of each chunk in seconds
#         
#     Returns:
#         np.ndarray: Extracted features
#     """
#     processor = Wav2Vec2Processor.from_pretrained('facebook/wav2vec2-base-960h')
#     model = Wav2Vec2Model.from_pretrained('facebook/wav2vec2-base-960h')
#     model.eval()
#     
#     chunk_size = int(chunk_duration * sample_rate)
#     total_samples = waveform.shape[1]
#     all_features = []

#     for start in range(0, total_samples, chunk_size):
#         end = min(start + chunk_size, total_samples)
#         chunk_waveform = waveform[:, start:end]
#         inputs = processor(
#             chunk_waveform.squeeze().numpy(), 
#             sampling_rate=sample_rate, 
#             return_tensors="pt", 
#             padding=True
#         )
#         with torch.no_grad():
#             features = model(**inputs).last_hidden_state
#             all_features.append(features.squeeze(0).numpy())

#     features_np = np.concatenate(all_features, axis=0)
#     return features_np


# def is_file_processed_w2v(connection, filename):
#     """Check if file has been processed with Wav2Vec2."""
#     cursor = connection.cursor()
#     query = "SELECT COUNT(*) FROM vector_audio_features WHERE filename = %s"
#     cursor.execute(query, (filename,))
#     result = cursor.fetchone()
#     return result[0] > 0


# def insert_file_path_into_db(connection, filename, features_file_path):
#     """Insert file path into database."""
#     cursor = connection.cursor()
#     query = """
#     INSERT INTO vector_audio_features (filename, features_file_path)
#     VALUES (%s, %s)
#     """
#     cursor.execute(query, (filename, features_file_path))
#     connection.commit()


# def save_features_to_file(features, filename):
#     """Save extracted features to a file."""
#     if not os.path.exists(FEATURES_DIR):
#         os.makedirs(FEATURES_DIR)
#     
#     features_file_path = os.path.join(FEATURES_DIR, f"{filename}_features.npy")
#     np.save(features_file_path, features)
#     return features_file_path


# def process_and_store_features(merged_df):
#     """Process dataset and extract Wav2Vec2 features."""
#     connection = connect_to_db()

#     for index, row in merged_df.iterrows():
#         audio_file_path = row['Filepath']
#         filename = row['Filename']
#         
#         # Check if the file is already processed
#         if is_file_processed_w2v(connection, filename):
#             print(f"File {filename} has already been processed. Skipping.")
#             continue
#         
#         try:
#             # Preprocess and extract features
#             waveform, sample_rate = preprocess_audio(audio_file_path)
#             wav2vec2_features = extract_wav2vec2_features(
#                 waveform, 
#                 sample_rate, 
#                 chunk_duration=30
#             )
#             
#             print(f"Feature vector size for {filename}: {wav2vec2_features.shape}")
#             
#             # Save the extracted features to a file
#             features_file_path = save_features_to_file(wav2vec2_features, filename)
#             
#             # Store the file path in the database
#             insert_file_path_into_db(connection, filename, features_file_path)
#             print(f"Features for {filename} stored successfully in {features_file_path}.")

#         except Exception as e:
#             print(f"Error processing {audio_file_path}: {e}")

#     connection.close()


# # Usage
# # process_and_store_features(merged_df)

print("Wav2Vec2 extraction functions available (commented out)")

## 15. New Extraction Capabilities Summary

This notebook now includes **BOTH** feature loading AND real-time extraction capabilities:

### Added Extraction Functions:

1. **Enhanced Audio Preprocessing** (Section 4b)
   - `preprocess_audio()` - Converts to WAV and resamples to 16kHz
   - Essential for consistent input to deep learning models

2. **Whisper Real-Time Extraction** (Section 9b)
   - `extract_whisper_features()` - Extracts speech embeddings from audio
   - `process_and_store_whisper_features()` - Main processing pipeline
   - Saves features as `.npy` files for later use

3. **OpenSmile Real-Time Extraction** (Section 10b)
   - `insert_features_into_db_opensmile()` - Stores 88 acoustic features as JSON
   - Processing loop for extracting eGeMAPSv02 features
   - Checks database to avoid re-processing

4. **Librosa Real-Time Extraction** (Section 11b)
   - `process_audio_files_librosa()` - Extracts 38 acoustic features:
     - 13 MFCCs (mel-frequency cepstral coefficients)
     - 12 Chroma features (pitch class profiles)
     - 7 Spectral contrast bands
     - 6 Tonnetz features (tonal centroids)
   - Stores features in individual database columns

5. **Alternative Trill Processing** (Section 12b)
   - Support for `*_std_trill_embedding.npy` file pattern
   - `load_numpy_to_columns()` - Expands 1024-dim embeddings into columns
   - Better file matching and validation

6. **Wav2Vec2 Extraction** (Section 14 - Optional)
   - Complete pipeline for Facebook's Wav2Vec2 model
   - Chunking strategy for long audio files
   - Currently commented out but ready to use

### Usage:

All extraction code is **commented out by default** to prevent accidental long-running operations. To use:

1. Locate the relevant section (e.g., "9b. Whisper Real-Time Feature Extraction")
2. Uncomment the processing code
3. Run the cell to extract features from your audio files
4. Features will be saved to disk and/or database

### Key Benefit:

You can now:
- **Extract** features from NEW audio files
- **Re-extract** features with different parameters
- **Load** existing pre-computed features for analysis
- Mix and match extraction and analysis as needed

In [ ]:
# Process OpenSmile features for database insertion
if len(opensmile_df) > 0:
    print("Processing OpenSmile features...")
    
    # Rename and clean
    opensmile_result = opensmile_df.rename(columns={'filename': 'group_id'})
    
    # Drop non-feature columns
    cols_to_drop = ['cel_total', 'id', 'created_at', 'opensmile_processed']
    opensmile_result = opensmile_result.drop(columns=[c for c in cols_to_drop if c in opensmile_result.columns])
    
    # Melt to long format
    opensmile_melted = opensmile_result.melt(
        id_vars=['group_id'],
        var_name='feat',
        value_name='value'
    )
    
    print(f"OpenSmile features melted: {len(opensmile_melted)} records")
    print(f"Unique features: {opensmile_melted['feat'].nunique()}")
    
    opensmile_melted.head()

In [ ]:
# Insert OpenSmile features into database
if 'opensmile_melted' in locals() and len(opensmile_melted) > 0:
    print("Inserting OpenSmile features into database...")
    
    try:
        connection = connect_to_db()
        insert_feature_table_into_db(connection, opensmile_melted, DB_TABLES['opensmile'])
        connection.close()
        print("OpenSmile features inserted successfully")
        
    except Exception as e:
        print(f"Error inserting OpenSmile features: {e}")
else:
    print("No OpenSmile features to insert")

## 11. Librosa Feature Extraction

In [ ]:
# Check Librosa processing status
def is_file_processed_librosa(filename, connection):
    """
    Check if file has been processed with Librosa.
    
    Args:
        filename (str): Filename to check
        connection: MySQL connection
        
    Returns:
        bool: True if processed
    """
    query = "SELECT COUNT(*) FROM new_audio_processing_status WHERE filename = %s AND librosa_processed = 1"
    cursor = connection.cursor()
    cursor.execute(query, (filename,))
    result = cursor.fetchone()
    cursor.close()
    return result[0] > 0

print("Librosa feature extraction setup complete")

In [ ]:
# Retrieve existing Librosa features from database
def retrieve_features_from_librosa_db():
    """
    Retrieve Librosa features from database.
    
    Returns:
        pd.DataFrame: Features dataframe
    """
    connection = connect_to_db()
    
    query = "SELECT * FROM new_audio_processing_status WHERE librosa_processed = 1"
    df = pd.read_sql(query, connection)
    
    connection.close()
    return df

try:
    librosa_df = retrieve_features_from_librosa_db()
    print(f"Retrieved {len(librosa_df)} Librosa feature records from database")
    
    if len(librosa_df) > 0:
        print(f"Sample columns: {list(librosa_df.columns[:10])}...")
        
except Exception as e:
    print(f"Could not retrieve Librosa features: {e}")
    librosa_df = pd.DataFrame()

In [ ]:
# Process Librosa features for database insertion
if len(librosa_df) > 0:
    print("Processing Librosa features...")
    
    librosa_result = librosa_df.rename(columns={'filename': 'group_id'})
    
    # Drop non-feature columns
    cols_to_drop = ['cel_total', 'created_at', 'librosa_processed']
    librosa_result = librosa_result.drop(columns=[c for c in cols_to_drop if c in librosa_result.columns])
    
    # Keep id column, melt the rest
    id_vars = ['id', 'group_id'] if 'id' in librosa_result.columns else ['group_id']
    
    librosa_melted = librosa_result.melt(
        id_vars=id_vars,
        var_name='feat',
        value_name='value'
    )
    
    print(f"Librosa features melted: {len(librosa_melted)} records")
    print(f"Unique features: {librosa_melted['feat'].nunique()}")
    
    librosa_melted.head()

In [ ]:
# Insert Librosa features into database
if 'librosa_melted' in locals() and len(librosa_melted) > 0:
    print("Inserting Librosa features into database...")
    
    try:
        connection = connect_to_db()
        insert_feature_table_into_db(connection, librosa_melted, DB_TABLES['librosa'])
        connection.close()
        print("Librosa features inserted successfully")
        
    except Exception as e:
        print(f"Error inserting Librosa features: {e}")
else:
    print("No Librosa features to insert")

## 12. Trill Feature Processing

In [ ]:
print(f"Scanning Trill features in {TRILL_FEATURES_DIR}...")

# Find all Trill embedding files
trill_files = list(TRILL_FEATURES_DIR.glob('*_wb_mean_trill_embedding.npy'))
print(f"Found {len(trill_files)} Trill feature files")

if len(trill_files) > 0:
    # Create mapping from filename to Trill path
    trill_mapping = {}
    
    for trill_file in trill_files:
        # Extract base filename (remove suffix)
        basename = trill_file.stem.replace('_wb_mean_trill_embedding', '')
        trill_mapping[basename] = str(trill_file)
    
    # Add to main dataframe
    if 'Filename' in feature_df.columns:
        feature_df['Trill_Path'] = feature_df['Filename'].map(trill_mapping)
        feature_df['Trill_Processed'] = feature_df['Trill_Path'].notna()
        
        print(f"Trill features available for {feature_df['Trill_Processed'].sum()} files")
    
    print(f"\nExample Trill file: {trill_files[0].name}")
else:
    print("No Trill features found")

In [ ]:
# Load and process Trill features
if 'Trill_Path' in feature_df.columns:
    print("Processing Trill features...")
    
    trill_data = []
    
    for idx, row in feature_df[feature_df['Trill_Processed'] == True].iterrows():
        filepath = row['Trill_Path']
        filename = row['Filename']
        klaatch_id = row.get('KlaatchID', 'unknown')
        
        if filepath and os.path.exists(filepath):
            try:
                embedding = np.load(filepath)
                
                # Trill embeddings are typically already mean-pooled
                if embedding.ndim > 1:
                    embedding = np.mean(embedding, axis=0)
                
                trill_data.append({
                    'group_id': filename,
                    'klaatch_id': klaatch_id,
                    'embedding': embedding
                })
                
            except Exception as e:
                print(f"Error loading {filepath}: {e}")
    
    if trill_data:
        print(f"Loaded Trill features for {len(trill_data)} files")
        
        # Create melted dataframe
        trill_records = []
        for record in trill_data:
            for i, val in enumerate(record['embedding']):
                trill_records.append({
                    'group_id': record['group_id'],
                    'feat': f'trill_mean_{i}',
                    'value': float(val)
                })
        
        trill_melted_df = pd.DataFrame(trill_records)
        print(f"Trill features melted: {len(trill_melted_df)} records")
        print(f"Embedding dimension: {trill_melted_df['feat'].nunique()}")
        
        trill_melted_df.head()
    else:
        print("No Trill features loaded")
else:
    print("No Trill path information available")

In [ ]:
# Insert Trill features into database
if 'trill_melted_df' in locals() and len(trill_melted_df) > 0:
    print("Inserting Trill features into database...")
    
    try:
        connection = connect_to_db()
        insert_feature_table_into_db(connection, trill_melted_df, DB_TABLES['trill'])
        connection.close()
        print("Trill features inserted successfully")
        
    except Exception as e:
        print(f"Error inserting Trill features: {e}")
else:
    print("No Trill features to insert")

## 13. Summary and Validation

In [ ]:
print("="*80)
print("PIPELINE SUMMARY")
print("="*80)

print(f"\nTotal records in dataset: {len(feature_df)}")

if 'Filepath' in feature_df.columns:
    print(f"Records with audio files: {feature_df['Filepath'].notna().sum()}")

if 'Whisper_Processed' in feature_df.columns:
    print(f"Records with Whisper features: {feature_df['Whisper_Processed'].sum()}")

if 'Trill_Processed' in feature_df.columns:
    print(f"Records with Trill features: {feature_df['Trill_Processed'].sum()}")

if 'Text_Length' in feature_df.columns:
    print(f"Records with transcripts: {(feature_df['Text_Length'] > 0).sum()}")

print(f"\nUnique participants (KlaatchID): {feature_df['KlaatchID'].nunique()}")

print("\n" + "="*80)
print("Pipeline execution complete")
print("="*80)

In [ ]:
# Display sample of final dataset
display_cols = ['Filename', 'KlaatchID', 'Age', 'CEL Total', 'Text_Length']
display_cols = [c for c in display_cols if c in feature_df.columns]

if display_cols:
    print("Sample of processed dataset:")
    feature_df[display_cols].head(10)